In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm
from scipy import stats

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from sqlalchemy import create_engine
import os

def connect_to_db():
    engine = create_engine(
        f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
        f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
    )
    return engine

engine = connect_to_db()

conn = engine.connect()

In [ ]:
df = pd.read_sql("SELECT * FROM public.gold_model_dataset", engine)

df.head()

In [ ]:
group_A = df[df["is_mobile"] == 0]["is_booking"]  # Desktop
group_B = df[df["is_mobile"] == 1]["is_booking"]  # Mobile

In [ ]:
print("Desktop:", group_A.mean())
print("Mobile:", group_B.mean())

In [ ]:
t_stat, p_value = stats.ttest_ind(group_A, group_B)

print("T-stat:", t_stat)
print("P-value:", p_value)

In [ ]:
alpha = 0.05

if p_value < alpha:
    print("Statistically significant difference")
else:
    print("No significant difference")

In [ ]:
lift = (group_B.mean() - group_A.mean()) / group_A.mean()
print("Lift:", lift)

In [ ]:
df[df["is_package"] == 1]["is_booking"]
df[df["is_package"] == 0]["is_booking"]

In [ ]:
df[df["distance_group"] == "near"]["is_booking"]
df[df["distance_group"] == "far"]["is_booking"]

In [ ]:
df.groupby(["is_mobile", "trip_type"])["is_booking"].mean()

In [ ]:
count = [group_A.sum(), group_B.sum()]
nobs = [len(group_A), len(group_B)]

stat, pval = proportions_ztest(count, nobs)

print(stat, pval)

In [ ]:
print("Desktop:", group_A.mean())
print("Mobile:", group_B.mean())
lift = (group_B.mean() - group_A.mean()) / group_A.mean()
print("Lift:", lift)

In [ ]:
df.groupby("is_mobile")["is_distance_unknown"].mean()

In [ ]:
df.groupby("is_mobile")["cnt"].mean()

In [ ]:
df.groupby("is_mobile")[["cnt", "advance_booking_days", "is_distance_unknown"]].mean()

In [ ]:
conn.close()

## A/B Test Analysis – Mobile vs Desktop

An observational A/B test was conducted to compare booking rates between mobile and desktop users.

- Desktop booking rate: 8.34%
- Mobile booking rate: 6.06%
- Relative lift: -27.38%

A two-sample statistical test showed a highly significant difference (p < 0.001), indicating that the observed gap is unlikely due to random variation.

### Interpretation

Mobile users exhibit a significantly lower conversion rate compared to desktop users. The effect size is substantial, with mobile showing approximately 27% lower conversion.

Further analysis controlling for key variables such as session intensity (`cnt`), booking window (`advance_booking_days`), and missing distance information (`is_distance_unknown`) indicates that this difference is not explained by these factors.

### Conclusion

The lower conversion rate on mobile appears to be a structural effect, potentially driven by differences in user behavior, context, or user experience across devices.

### Limitations

This analysis is based on observational data and does not represent a randomized controlled experiment. Therefore, causal conclusions cannot be definitively established.